# Z01 policy playback

Select a GPU runtime, run setup, then upload a Zlab Z01 walking or BallSprint checkpoint.
The second cell displays the rollout as a 30 FPS MuJoCo video.


In [ ]:

#@title 1. Set up MuJoCo playback
import os
from pathlib import Path
import subprocess
import sys

repo_dir = Path("/content/Z01-Colab-Playback")
runtime_dir = Path("/content/z01_playback")
runtime_dir.mkdir(parents=True, exist_ok=True)

if (repo_dir / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/hafizadha/Z01-Colab-Playback.git", str(repo_dir)],
        check=True,
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo_dir / "requirements.txt")],
    check=True,
)

run_env = {
    **os.environ,
    "MUJOCO_GL": "egl",
    "PYOPENGL_PLATFORM": "egl",
    "WARP_CACHE_PATH": str(runtime_dir / "warp-cache"),
    "MJLAB_WARP_QUIET": "1",
    "OMP_NUM_THREADS": "2",
}
renderer_probe = """
import mujoco
model = mujoco.MjModel.from_xml_string('<mujoco/>')
with mujoco.Renderer(model, height=64, width=64) as renderer:
    renderer.update_scene(mujoco.MjData(model))
    assert renderer.render().shape == (64, 64, 3)
"""
probe = subprocess.run([sys.executable, "-c", renderer_probe], env=run_env, capture_output=True)
if probe.returncode:
    print("EGL is unavailable. Setting up MuJoCo software rendering.", flush=True)
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "libosmesa6"], check=True)
    run_env.update(MUJOCO_GL="osmesa", PYOPENGL_PLATFORM="osmesa")
    subprocess.run([sys.executable, "-c", renderer_probe], env=run_env, check=True)
print("Ready. Run the next cell to upload a checkpoint.")


In [ ]:
#@title 2. Upload a checkpoint and play
POLICY = "RobustWalk2" #@param ["Walk", "FastWalk", "StrideWalk", "LongStrideWalk", "RobustWalk1", "RobustWalk2", "RobustWalk3", "BalanceRecovery", "BallSprint"]
#@markdown **TARGET_SPEED** applies to walking policies. BallSprint sets its speed from the ball position.
TARGET_SPEED = 1.55 #@param {type:"number"}

from google.colab import files
from IPython.display import Video, display

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one Zlab .pt checkpoint.")
name, contents = uploaded.popitem()
if Path(name).suffix.lower() != ".pt":
    raise ValueError("Choose a .pt checkpoint.")
checkpoint_dir = runtime_dir / "checkpoint"
checkpoint_dir.mkdir(exist_ok=True)
checkpoint = checkpoint_dir / Path(name).name
checkpoint.write_bytes(contents)
del contents

video_path = runtime_dir / "z01_playback.mp4"
video_path.unlink(missing_ok=True)
subprocess.run(
    [sys.executable, "-u", "playback.py", str(checkpoint),
     POLICY, str(TARGET_SPEED), str(video_path)],
    cwd=repo_dir, env=run_env, check=True,
)
display(Video(str(video_path), embed=True))
